<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week8/Day3/Exercices_XP/Exercises_XP_MCP_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: Minimal MCP over STDIO (Student)

Build a tiny MCP server and client that talk over STDIO. This code is supposed to be executed in a local jupyter notebook not Colab's notebook.

## What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is great locally.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

## Setup
Run the install cell, then restart the runtime if Colab asks. Python 3.10+ required.

In [1]:
# Install MCP CLI + SDK
%pip install -qU "mcp[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 7.6 MB/s eta 0:00:00


In [2]:
# Quick verify
!python --version
!mcp --help | head -n 5

Python 3.12.13
                                                                                
 Usage: mcp [OPTIONS] COMMAND [ARGS]...                                         
                                                                                
 MCP development tools                                                          
                                                                                


## A. Server (server.py)
Create a small MCP server named "Demo" with:
- Tool `add(a: int, b: int) -> int` returning the sum.
- Resource template `greeting://{name}` returning "Hello, {name}!".
- Start the STDIO loop in `__main__`.

In [6]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    # Start the server loop over STDIO
    mcp.run()

Overwriting server.py


## B. Client (client.py)
Write a client that:
1) Spawns the server via STDIO using the MCP CLI.
2) Initializes a session.
3) Lists resources and tools, printing their names.
4) Reads `greeting://hello` and prints it.
5) Calls tool `add` with a=1, b=7 and prints the result.

In [7]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)

def extract_content(payload):
    """Best-effort to pull text from MCP responses."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # list resources and print their URIs
            resources = await session.list_resources()
            print("Resources:")
            for res in resources.resources:
                print(f" - {res.uri}")

            # list tools and print their names
            tools = await session.list_tools()
            print("\nTools:")
            for tool in tools.tools:
                print(f" - {tool.name}")

            # read greeting://hello and print the content
            greeting = await session.read_resource("greeting://hello")
            print(f"\nResource content (greeting://hello): {extract_content(greeting)}")

            # call add with a=1, b=7 and print the result
            result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print(f"\nTool call result (add 1+7): {extract_content(result)}")

if __name__ == "__main__":
    asyncio.run(run())

Overwriting client.py


## C. Run
One terminal (client spawns server):
```
python client.py
```

Or two terminals:
```
mcp run server.py
python client.py
```

In Colab, run the next cell (client will spawn the server automatically).

In [9]:
# Run the client (spawns the server over STDIO)
!python client.py

[07/08/26 09:52:53] INFO     Processing request of type            ]8;id=577585;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=341553;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
Resources:
                    INFO     Processing request of type            ]8;id=594744;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=851692;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   

Tools:
 - add
                    INFO     Processing request of type            ]8;id=925497;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=578939;file:///usr/local/lib/python3.12/dist-packages/mcp/serve

In [8]:
!python client.py

[07/08/26 09:52:48] INFO     Processing request of type            ]8;id=624570;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=568138;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListResourcesRequest                               
Resources:
                    INFO     Processing request of type            ]8;id=788951;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=125663;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   

Tools:
 - add
                    INFO     Processing request of type            ]8;id=754746;file:///usr/local/lib/python3.12/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=430460;file:///usr/local/lib/python3.12/dist-packages/mcp/serve

## Troubleshooting
- `mcp: command not found` ? rerun the install cell or restart runtime.
- Connection closed ? open a second terminal and run `mcp run server.py` to check server errors.
- Type errors ? ensure JSON args are ints for `add`.